In [0]:
def read_csv_df(spark,path,header=True,infer_schema=True,sep=","):
    return (spark.read
        .option("header", header)
        .option("inferSchema", infer_schema)
        .option("sep", sep)
        .csv(path))

df1=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source1.txt',True,True,',')
df1.display()
df2=read_csv_df(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_source2.txt',True,True,',')
df2.display()

In [0]:
#Select (Projection)

df1 = df1.select(df1.first_name, df1.role)
df2 = df2.select(df2.first_name,df2.hub_location )
finaldf = df1.join(df2, on='first_name', how='inner')
finaldf.display()


In [0]:
from pyspark.sql.functions import concat,lit
def read_json(apark,path,mline):
    df = spark.read.json(path, multiLine=mline)
    return df

jsondf=read_json(spark,'/Volumes/logistics_catalog_assign/landing_zone/landing_vol/logistics_shipment_detail_3000.json',True)

In [0]:
##Filter (Selection)
jsondf = jsondf.filter((jsondf.shipment_status == 'DELAYED') | (jsondf.shipment_status == 'RETURNED'))
jsondf.display()
jsondf = jsondf.filter(jsondf.shipment_weight_kg > 500)
jsondf.display()

In [0]:
#Derive Flags & Columns (Business Logic)
from pyspark.sql.functions import when, dayofweek, to_date

jsondf = jsondf.withColumn('is_high_value', when(jsondf.shipment_cost > 50000, lit('True')).otherwise(lit('False')))
jsondf.display()

jsondf = jsondf.withColumn('is_weekend', when(dayofweek(to_date(jsondf.shipment_date, 'yy-MM-dd')).isin([1,7]), 'True').otherwise('False'))
jsondf.display()


In [0]:
#Format (Standardization)
from pyspark.sql.functions import concat, upper, col

jsondf = jsondf.withColumn('shipment_cost', concat(lit('$'), jsondf.shipment_cost.cast('string')))
jsondf.display()


In [0]:
jsondf = jsondf.withColumn('source_city', upper(col("source_city")))
jsondf.display()

In [0]:
#Group & Aggregate (Summarization)
df2 =  df2.groupBy('hub_location').count()
df2.display()


In [0]:
jsondf = jsondf.groupBy(jsondf.vehicle_type).agg({'shipment_weight_kg': 'sum'})
jsondf.display()

In [0]:
#Sorting (Ordering)
jsondf = jsondf.orderBy(jsondf.shipment_cost, ascending=False)
jsondf.display()

In [0]:
jsondf = jsondf.orderBy(jsondf.shipment_date, ascending=True)
jsondf.display()


In [0]:
#Limit (Top-N Analysis)
jsondf = jsondf.filter(jsondf.shipment_status == 'DELAYED').orderBy(jsondf.shipment_cost,ascending=True).limit(10)
jsondf.display()